# LeetCode 1011 — Capacity To Ship Packages Within D Days bằng Python

Link đề: https://leetcode.com/problems/capacity-to-ship-packages-within-d-days/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (sức chứa nhỏ nhất). Điểm cốt lõi: hàm "số ngày cần với sức chứa `c`" là **không tăng** theo `c` ⇒ **tìm nhị phân trên đáp án**; mỗi lần kiểm tra bằng tham lam một lượt. Kiểm chứng độc lập bằng **quy hoạch động chia mảng** (tối thiểu hoá tổng đoạn lớn nhất).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia dãy `weights` (giữ thứ tự) thành **tối đa `days` đoạn liên tiếp**; sức chứa `c` phải ≥ tổng mỗi đoạn; tìm `c` nhỏ nhất.
2. Nắm công cụ: **tìm nhị phân trên đáp án** (binary search on answer), kiểm tra khả thi bằng tham lam, `bisect` trên tổng tiền tố, quy hoạch động tách mảng.
3. Đi từ **thử tăng dần sức chứa** → **tìm nhị phân + kiểm tra tham lam** → **kiểm tra bằng bước nhảy `bisect` trên tổng tiền tố** → **tìm kiếm mũ (galloping) rồi nhị phân** → **quy hoạch động tách mảng**.
4. Hiểu mấu chốt: `feasible(c)` = "đủ trong `days` ngày với sức chứa `c`" **đơn điệu** (nếu `c` đủ thì mọi `c' ≥ c` đủ); biên: `max(weights) ≤ đáp án ≤ sum(weights)`.
5. Tránh lỗi kinh điển: cận dưới là `1` hoặc trung bình (phải là `max(weights)`), đếm ngày lệch 1 (quên ngày đầu), dùng `<` thay `<=` khi kiểm tra tải, sắp xếp lại `weights` (không được đổi thứ tự), bỏ mất gói khi mở ngày mới, vòng nhị phân sai biên (`mid` vs `mid ± 1`).

## 2. Đề bài & ví dụ

Băng chuyền có các gói hàng cần vận chuyển từ cảng này sang cảng khác trong vòng `days` ngày.

Gói thứ `i` nặng `weights[i]`. Mỗi ngày ta chất lên tàu các gói **theo thứ tự trên băng chuyền** (liên tiếp), không được vượt quá **sức chứa tối đa** của tàu.

Trả về sức chứa **nhỏ nhất** của tàu để vận chuyển hết các gói trong vòng `days` ngày.

**Ví dụ 1:** `weights = [1,2,3,4,5,6,7,8,9,10]`, `days = 5` → `15`.

**Ví dụ 2:** `weights = [3,2,2,4,1,4]`, `days = 3` → `6`.

**Ví dụ 3:** `weights = [1,2,3,1,1]`, `days = 4` → `3`.

**Ràng buộc:** `1 <= days <= weights.length <= 5·10^4`; `1 <= weights[i] <= 500`.

In [1]:
import bisect
import itertools
import random
import time
from typing import List

examples = [(([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), 15), (([3, 2, 2, 4, 1, 4], 3), 6), (([1, 2, 3, 1, 1], 4), 3), (([5], 1), 5), (([1, 1, 1], 1), 3)]
for (w, d), want in examples:
    print(f"weights = {w}, days = {d} -> {want}")

weights = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], days = 5 -> 15
weights = [3, 2, 2, 4, 1, 4], days = 3 -> 6
weights = [1, 2, 3, 1, 1], days = 4 -> 3
weights = [5], days = 1 -> 5
weights = [1, 1, 1], days = 1 -> 3


## 3. Kiến thức nền

### 3.1 Số ngày cần với sức chứa `c` (tham lam)

Với sức chứa `c` cố định, chất **càng nhiều càng tốt** mỗi ngày: duyệt các gói theo thứ tự, cộng vào ngày hiện tại nếu còn chỗ, ngược lại **mở ngày mới**. Số ngày thu được là **nhỏ nhất** có thể (mục 10 chứng minh). Nếu có gói nặng hơn `c` thì không thể (trả `∞`).

In [2]:
def days_needed(weights, cap):
    if max(weights) > cap:
        return float("inf")                           # một gói nặng hơn sức chứa: không bao giờ chở được
    days, load = 1, 0
    for w in weights:
        if load + w > cap:                            # không còn chỗ: sang ngày mới
            days += 1
            load = 0
        load += w
    return days


w = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print([(c, days_needed(w, c)) for c in (10, 14, 15, 20, 55)])

[(10, 7), (14, 6), (15, 5), (20, 4), (55, 1)]


### 3.2 Tính đơn điệu

`days_needed(w, c)` **không tăng** theo `c`: sức chứa lớn hơn không bao giờ cần nhiều ngày hơn. Vì thế tập `{c : days_needed(w, c) ≤ days}` là một **tia** `[c*, ∞)` và `c*` là đáp án.

### 3.3 Biên của đáp án

- Cận dưới: `max(weights)` (gói nặng nhất phải vào một ngày nào đó).
- Cận trên: `sum(weights)` (chở tất cả trong một ngày; luôn đủ vì `days ≥ 1`). Cận trên tốt hơn: `max(weights) + ceil(sum/days)`... (không cần).

In [3]:
w, d = [3, 2, 2, 4, 1, 4], 3
lo, hi = max(w), sum(w)
print("biên:", lo, hi, "| days_needed ở hai biên:", days_needed(w, lo), days_needed(w, hi))
assert days_needed(w, hi) == 1 and days_needed(w, lo - 1) == float("inf")

biên: 4 16 | days_needed ở hai biên: 5 1


## 4. Ý tưởng nền tảng / trực giác

`weights = [3,2,2,4,1,4]`, `days = 3`. Thử sức chứa `c`:

| `c` | các ngày (tham lam) | số ngày | đủ trong 3? |
|---|---|---|---|
| 4 | `[3] [2,2] [4] [1] [4]` | 5 | không |
| 5 | `[3,2] [2] [4,1] [4]` | 4 | không |
| 6 | `[3,2] [2,4] [1,4]` | 3 | **có** |

`c = 6` là nhỏ nhất đủ ⇒ đáp án `6`. Vì "đủ" đơn điệu, ta không cần thử từng `c`: **tìm nhị phân** trong `[max, sum]`.

In [4]:
def schedule(weights, cap):
    out, cur, load = [], [], 0
    for w in weights:
        if load + w > cap:
            out.append(cur)
            cur, load = [], 0
        cur.append(w)
        load += w
    out.append(cur)
    return out


for c in (4, 5, 6):
    print(c, schedule([3, 2, 2, 4, 1, 4], c), "->", len(schedule([3, 2, 2, 4, 1, 4], c)), "ngày")

4 [[3], [2, 2], [4], [1], [4]] -> 5 ngày
5 [[3, 2], [2], [4, 1], [4]] -> 4 ngày
6 [[3, 2], [2, 4], [1, 4]] -> 3 ngày


## 5. Brute force — thử tăng dần sức chứa

**Ý tưởng:** thử `c = max(weights), max + 1, …` cho đến khi `days_needed(c) ≤ days`. `O(n · (đáp án − max))` — với `n = 5·10^4` và hiệu tới `~2.5·10^7` là quá chậm, nhưng làm oracle cho mảng nhỏ.

In [5]:
def ship_scan(weights, days):
    c = max(weights)
    while days_needed(weights, c) > days:
        c += 1
    return c


for (w, d), want in examples:
    assert ship_scan(w, d) == want
print([ship_scan(w, d) for (w, d), _ in examples])

[15, 6, 3, 5, 3]


In [6]:
# Code nộp LeetCode cho Brute force (quá chậm với dữ liệu lớn)
class SolutionScan:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        def need(cap):
            d, load = 1, 0
            for w in weights:
                if load + w > cap:
                    d += 1
                    load = 0
                load += w
            return d

        c = max(weights)
        while need(c) > days:
            c += 1
        return c

**Độ phức tạp:** `O(n · Δ)` với `Δ = đáp án − max(weights)` (xấu nhất `~ n · 500 · n/days`).

**Nút thắt:** ta thử cả `Δ` giá trị liên tiếp trong khi tính đơn điệu cho phép bỏ qua một nửa mỗi lần.

## 6. Bước đột phá

Hàm `days_needed` đơn điệu không tăng theo sức chứa ⇒ tìm nhị phân giá trị nhỏ nhất `c` có `days_needed(c) ≤ days` trong `[max(weights), sum(weights)]`: `O(n · log(sum))`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** tìm nhị phân + tham lam (đáp án chính), **Cách 3** tìm nhị phân + bước nhảy `bisect`, **Cách 4** tìm kiếm mũ rồi nhị phân, **Cách 5** quy hoạch động tách mảng.

### Cách 2 — Tìm nhị phân trên đáp án

**Giải thích từng đoạn:**

1. `lo = max(weights)`, `hi = sum(weights)`.
2. Khi `lo < hi`: `mid = (lo + hi) // 2`; nếu `days_needed(mid) ≤ days` thì `hi = mid` (mid có thể là đáp án), ngược lại `lo = mid + 1`.
3. Trả về `lo`.

In [7]:
def ship_binary(weights, days):
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(weights, mid) <= days:         # mid đủ: đáp án nằm trong [lo, mid]
            hi = mid
        else:                                         # mid không đủ: đáp án > mid
            lo = mid + 1
    return lo


for (w, d), want in examples:
    assert ship_binary(w, d) == want
print([ship_binary(w, d) for (w, d), _ in examples])

[15, 6, 3, 5, 3]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionBinary:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        def need(cap):
            d, load = 1, 0
            for w in weights:
                if load + w > cap:
                    d += 1
                    load = 0
                load += w
            return d

        lo, hi = max(weights), sum(weights)
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= days:
                hi = mid
            else:
                lo = mid + 1
        return lo

**Độ phức tạp:** `O(n log(sum))` thời gian (`log(2.5·10^7) ≈ 25` lần kiểm tra, mỗi lần `O(n)`), `O(1)` bộ nhớ.

### Cách 3 — Tìm nhị phân + bước nhảy `bisect` trên tổng tiền tố

**Ý tưởng:** với tổng tiền tố `P` (`P[0] = 0`), kiểm tra `c` bằng cách **nhảy**: ngày hiện tại bắt đầu ở chỉ số `i`, kết thúc ở chỉ số lớn nhất `j` có `P[j] ≤ P[i] + c` (`bisect_right(P, P[i] + c) − 1`); lặp cho tới `j = n` hoặc hết `days` ngày. Mỗi lần kiểm tra `O(days · log n)` thay vì `O(n)` — lợi khi `days` nhỏ.

In [9]:
def ship_prefix_bisect(weights, days):
    n = len(weights)
    P = [0]
    for w in weights:
        P.append(P[-1] + w)

    def feasible(cap):
        i = 0
        for _ in range(days):
            j = bisect.bisect_right(P, P[i] + cap) - 1           # chở được các gói i..j-1
            if j == i:                                           # không chở nổi cả gói i (nặng hơn cap)
                return False
            i = j
            if i == n:
                return True
        return i == n

    lo, hi = max(weights), P[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


for (w, d), want in examples:
    assert ship_prefix_bisect(w, d) == want
print([ship_prefix_bisect(w, d) for (w, d), _ in examples])

[15, 6, 3, 5, 3]


In [10]:
# Code nộp LeetCode cho Cách 3
import bisect


class SolutionPrefixBisect:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        n = len(weights)
        P = [0]
        for w in weights:
            P.append(P[-1] + w)

        def feasible(cap):
            i = 0
            for _ in range(days):
                j = bisect.bisect_right(P, P[i] + cap) - 1
                if j == i:
                    return False
                i = j
                if i == n:
                    return True
            return i == n

        lo, hi = max(weights), P[-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if feasible(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

**Độ phức tạp:** `O(n + days · log n · log(sum))` thời gian, `O(n)` bộ nhớ (tổng tiền tố).

### Cách 4 — Tìm kiếm mũ (galloping) rồi nhị phân

**Ý tưởng:** bắt đầu từ `c = max(weights)`; nhân đôi `c` cho đến khi đủ; đáp án nằm trong `(c/2, c]` (hoặc `[max, c]`); rồi nhị phân. Hữu ích khi biên trên khó biết trước.

In [11]:
def ship_galloping(weights, days):
    lo = max(weights)
    c = lo
    while days_needed(weights, c) > days:
        lo = c + 1                                  # c không đủ: đáp án >= c + 1
        c *= 2
    hi = c                                          # hi đủ
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(weights, mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


for (w, d), want in examples:
    assert ship_galloping(w, d) == want
print([ship_galloping(w, d) for (w, d), _ in examples])

[15, 6, 3, 5, 3]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionGalloping:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        def need(cap):
            d, load = 1, 0
            for w in weights:
                if load + w > cap:
                    d += 1
                    load = 0
                load += w
            return d

        lo = c = max(weights)
        while need(c) > days:
            lo = c + 1
            c *= 2
        hi = c
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= days:
                hi = mid
            else:
                lo = mid + 1
        return lo

**Độ phức tạp:** `O(n · log(đáp án))` thời gian, `O(1)` bộ nhớ.

### Cách 5 — Quy hoạch động tách mảng

**Ý tưởng:** `dp[d][i]` = tổng đoạn lớn nhất **nhỏ nhất** khi chia `i` gói đầu thành đúng `d` đoạn liên tiếp (không rỗng). `dp[1][i] = P[i]`; `dp[d][i] = min_{j < i} max(dp[d−1][j], P[i] − P[j])`. Đáp án `= min_{d ≤ days} dp[d][n]` (dùng ít ngày hơn cũng được; thực tế `dp` không tăng theo `d` nên bằng `dp[days][n]` khi `days ≤ n`). `O(days · n²)` — chỉ cho mảng nhỏ; là **oracle** độc lập (không dùng tham lam hay nhị phân).

In [13]:
def ship_dp(weights, days):
    n = len(weights)
    P = [0]
    for w in weights:
        P.append(P[-1] + w)
    INF = float("inf")
    dp = [[INF] * (n + 1) for _ in range(days + 1)]
    for i in range(1, n + 1):
        dp[1][i] = P[i]
    for d in range(2, days + 1):
        for i in range(d, n + 1):
            best = INF
            for j in range(d - 1, i):                          # j gói đầu cho d-1 ngày, các gói j..i-1 cho ngày d
                best = min(best, max(dp[d - 1][j], P[i] - P[j]))
            dp[d][i] = best
    return min(dp[d][n] for d in range(1, days + 1))


for (w, d), want in examples:
    assert ship_dp(w, d) == want
print([ship_dp(w, d) for (w, d), _ in examples])

[15, 6, 3, 5, 3]


In [14]:
# Code nộp LeetCode cho Cách 5 (chỉ minh hoạ: O(days·n^2))
class SolutionDP:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        n = len(weights)
        P = [0]
        for w in weights:
            P.append(P[-1] + w)
        INF = float("inf")
        dp = [[INF] * (n + 1) for _ in range(days + 1)]
        for i in range(1, n + 1):
            dp[1][i] = P[i]
        for d in range(2, days + 1):
            for i in range(d, n + 1):
                dp[d][i] = min(max(dp[d - 1][j], P[i] - P[j]) for j in range(d - 1, i))
        return min(dp[d][n] for d in range(1, days + 1))

**Độ phức tạp:** `O(days · n²)` thời gian, `O(days · n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `weights = [3,2,2,4,1,4]`, `days = 3`: `lo = 4`, `hi = 16`.

| vòng | `lo` | `hi` | `mid` | `days_needed(mid)` | quyết định |
|---|---|---|---|---|---|
| 1 | 4 | 16 | 10 | 2 (`[3,2,2] [4,1,4]`) | đủ ⇒ `hi = 10` |
| 2 | 4 | 10 | 7 | 3 (`[3,2,2] [4,1] [4]`) | đủ ⇒ `hi = 7` |
| 3 | 4 | 7 | 5 | 4 | không đủ ⇒ `lo = 6` |
| 4 | 6 | 7 | 6 | 3 | đủ ⇒ `hi = 6` |

`lo = hi = 6` ⇒ đáp án `6`. Hàm trace in từng vòng:

In [15]:
def trace_binary(weights, days):
    lo, hi = max(weights), sum(weights)
    print(f"weights = {weights}, days = {days}: lo = {lo}, hi = {hi}")
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        need = days_needed(weights, mid)
        ok = need <= days
        print(f"  vòng {step}: lo={lo} hi={hi} mid={mid}: {schedule(weights, mid)} -> {need} ngày -> {'đủ, hi = mid' if ok else 'thiếu, lo = mid + 1'}")
        if ok:
            hi = mid
        else:
            lo = mid + 1
    print("=>", lo)
    return lo


assert trace_binary([3, 2, 2, 4, 1, 4], 3) == 6

weights = [3, 2, 2, 4, 1, 4], days = 3: lo = 4, hi = 16
  vòng 1: lo=4 hi=16 mid=10: [[3, 2, 2], [4, 1, 4]] -> 2 ngày -> đủ, hi = mid
  vòng 2: lo=4 hi=10 mid=7: [[3, 2, 2], [4, 1], [4]] -> 3 ngày -> đủ, hi = mid
  vòng 3: lo=4 hi=7 mid=5: [[3, 2], [2], [4, 1], [4]] -> 4 ngày -> thiếu, lo = mid + 1
  vòng 4: lo=6 hi=7 mid=6: [[3, 2], [2, 4], [1, 4]] -> 3 ngày -> đủ, hi = mid
=> 6


**Trace biên dưới:** `weights = [5]`, `days = 1`: `lo = hi = 5` nên trả `5` ngay (không cần vòng lặp). Nếu đặt `lo = 1`, nhị phân có thể thử `mid = 3 < 5`: nếu hàm đếm ngày vẫn "nhét" gói nặng `5` vào một ngày trống thì nó báo `1 ngày` ⇒ đủ ⇒ sức chứa `3` bị chấp nhận sai (xem mục 13).

## 9. Code nộp LeetCode

Chọn **Cách 2** (tìm nhị phân + tham lam). Giữ nguyên `class Solution`, `shipWithinDays(self, weights, days)`:

In [16]:
class Solution:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        def need(cap):                                 # số ngày cần với sức chứa cap (cap >= max(weights))
            d, load = 1, 0
            for w in weights:
                if load + w > cap:                     # không chứa thêm được: sang ngày mới
                    d += 1
                    load = 0
                load += w
            return d

        lo, hi = max(weights), sum(weights)            # đáp án nằm trong [max, sum]
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= days:
                hi = mid                               # mid đủ: thử nhỏ hơn
            else:
                lo = mid + 1                           # mid thiếu: phải lớn hơn
        return lo


print([Solution().shipWithinDays(list(w), d) for (w, d), _ in examples])
assert all(Solution().shipWithinDays(list(w), d) == want for (w, d), want in examples)

[15, 6, 3, 5, 3]


## 10. Vì sao đúng

**(a) Tham lam tính số ngày tối thiểu.** Với sức chứa `c ≥ max(weights)`, gọi `g(c)` là số ngày của tham lam và `OPT(c)` là số ngày tối thiểu của mọi cách chia mảng thành đoạn liên tiếp có tổng `≤ c`. Chứng minh `g(c) = OPT(c)` bằng "tham lam đi trước": giả sử một cách chia tối ưu có các điểm cắt `q₁ < q₂ < …`; tham lam có các điểm cắt `p₁ < p₂ < …`. Quy nạp: `p_k ≥ q_k` với mọi `k` (đoạn thứ `k` của tham lam kết thúc xa nhất có thể: nếu `p_{k−1} ≥ q_{k−1}` thì đoạn bắt đầu tại `p_{k−1}` chứa được ít nhất các phần tử mà đoạn tối ưu bắt đầu tại `q_{k−1}` chứa, vì bắt đầu muộn hơn thì tổng bớt đi). Do đó tham lam không cần nhiều đoạn hơn tối ưu.

**(b) Tính đơn điệu.** Nếu `c ≤ c'` thì mọi cách chia hợp lệ với `c` cũng hợp lệ với `c'`; nên `g(c') ≤ g(c)` và tập `{c : g(c) ≤ days}` là tia `[c*, ∞)`.

**(c) Biên.** Mọi `c` hợp lệ phải `≥ max(weights)` (gói nặng nhất nằm trong một đoạn). `c = sum(weights)` cho `g = 1 ≤ days`. Vậy `c* ∈ [max, sum]`.

**(d) Tìm nhị phân.** Bất biến: `c* ∈ [lo, hi]`. Nếu `mid` đủ thì `c* ≤ mid` ⇒ `hi = mid`; nếu thiếu thì `c* > mid` ⇒ `lo = mid + 1`. Khoảng thu hẹp tới `lo = hi = c*`. `mid ≥ lo ≥ max(weights)` nên `need(mid)` luôn xác định (không có gói quá nặng).

**(e) Cách 3.** `bisect_right(P, P[i] + cap) − 1` là chỉ số lớn nhất `j` với `P[j] ≤ P[i] + cap`, tức tổng gói `i..j−1` `≤ cap`: đúng là "chất đầy" một ngày bắt đầu từ `i`. Nếu `j = i` thì gói `i` nặng hơn `cap` ⇒ không khả thi. `days` bước nhảy phải phủ hết `n` gói.

**(f) Cách 4.** Nhân đôi `c` cho tới khi đủ: sau đó `c* ∈ [lo, c]` với `lo = (giá trị thiếu cuối) + 1`; rồi nhị phân.

**(g) Cách 5.** `dp[d][i]` là giá trị cực tiểu của tổng đoạn lớn nhất khi chia `i` gói đầu thành `d` đoạn liên tiếp: đoạn cuối là gói `j..i−1`, phần đầu là `dp[d−1][j]`; lấy `min` trên `j`. Sức chứa tối thiểu = tổng đoạn lớn nhất tối thiểu trên mọi cách chia thành `≤ days` đoạn: chính là đáp án (một cách chia với tổng đoạn lớn nhất `M` ⇔ sức chứa `M` đủ).

**(h) Kiểm chứng bằng máy.** Mọi cách khớp oracle DP (độc lập) trên **mọi** mảng `n ≤ 6` với giá trị trong `[1, 4]` và mọi `days ∈ [1, n]`; khớp nhau tới `n = 5·10^4`; đáp án thoả `days_needed(ans) ≤ days` và `days_needed(ans − 1) > days` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử tăng dần | `O(n·Δ)` | `O(1)` | oracle mảng nhỏ |
| 2. Nhị phân + tham lam | `O(n log S)` | `O(1)` | **đáp án chính** |
| 3. Nhị phân + `bisect` | `O(n + days·log n·log S)` | `O(n)` | `days` nhỏ |
| 4. Galloping + nhị phân | `O(n log ans)` | `O(1)` | không cần biên trên |
| 5. DP tách mảng | `O(days·n²)` | `O(days·n)` | oracle độc lập, `n` nhỏ |

(`S = sum(weights)`.) Đo thực tế `n = 5·10^4`:

In [17]:
def bench(fn, args, repeat=2):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1011)
big = [rng.randint(1, 500) for _ in range(50000)]
for days in (1, 10, 1000, 50000):
    print(f"days = {days}")
    for name, fn in [("binary", ship_binary), ("prefix_bisect", ship_prefix_bisect), ("galloping", ship_galloping)]:
        print(f"    {name:14s}: {bench(fn, (big, days)) * 1e3:8.1f} ms -> {fn(big, days)}")

days = 1


    binary        :     81.6 ms -> 12510150
    prefix_bisect :      3.7 ms -> 12510150


    galloping     :    140.8 ms -> 12510150
days = 10


    binary        :     88.8 ms -> 1251150
    prefix_bisect :      3.5 ms -> 1251150


    galloping     :    120.8 ms -> 1251150
days = 1000


    binary        :     85.1 ms -> 12678
    prefix_bisect :      8.0 ms -> 12678


    galloping     :     73.3 ms -> 12678
days = 50000


    binary        :     94.7 ms -> 500


    prefix_bisect :     76.3 ms -> 500
    galloping     :      5.2 ms -> 500


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle DP, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `n = 5·10^4`.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "binary": ship_binary,
    "prefix_bisect": ship_prefix_bisect,
    "galloping": ship_galloping,
    "scan": ship_scan,
    "dp": ship_dp,
    "Solution": lambda w, d: Solution().shipWithinDays(list(w), d),
    "SolutionBinary": lambda w, d: SolutionBinary().shipWithinDays(list(w), d),
    "SolutionPrefixBisect": lambda w, d: SolutionPrefixBisect().shipWithinDays(list(w), d),
    "SolutionGalloping": lambda w, d: SolutionGalloping().shipWithinDays(list(w), d),
    "SolutionScan": lambda w, d: SolutionScan().shipWithinDays(list(w), d),
    "SolutionDP": lambda w, d: SolutionDP().shipWithinDays(list(w), d),
}
SLOW = {"scan", "SolutionScan", "dp", "SolutionDP"}
fast = [n for n in IMPLS if n not in SLOW]


def valid_capacity(weights, days, cap):
    """Hàm kiểm tra tính tối ưu: cap đủ và cap - 1 không đủ (độc lập với việc tìm cap)."""
    return days_needed(weights, cap) <= days and days_needed(weights, cap - 1) > days


def check(w, d, want=None, slow_n=12):
    if want is None and len(w) <= slow_n:
        want = ship_dp(w, d)
    results = {}
    for name in IMPLS:
        if name in SLOW and len(w) > slow_n:
            continue
        results[name] = IMPLS[name](w, d)
    assert len(set(results.values())) == 1, (w[:8], d, results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (w[:8], d, got, want)
    assert valid_capacity(w, d, got), (w[:8], d, got)
    return got


for (w, d), want in examples:
    check(w, d, want)
print("ví dụ đề: oracle DP và", len(IMPLS), "cài đặt đúng; sức chứa thoả điều kiện tối ưu")

ví dụ đề: oracle DP và 11 cài đặt đúng; sức chứa thoả điều kiện tối ưu


### B. Edge case

- `n = 1`: đáp án là chính gói đó.
- `days = 1`: đáp án `sum(weights)`.
- `days = n`: đáp án `max(weights)`.
- Mọi gói bằng nhau.
- Một gói rất nặng và nhiều gói nhẹ (cận dưới `max` quyết định).
- Gói nhẹ trước, nặng sau; nặng trước, nhẹ sau.
- Trường hợp `sum/days` lớn hơn `max` (cận dưới trung bình quyết định).

In [19]:
edge = [
    ([5], 1), ([500], 1), ([1, 2, 3], 1), ([1, 2, 3], 3), ([4, 4, 4, 4], 2), ([4, 4, 4, 4], 4), ([1, 1, 1, 1, 100], 2), ([100, 1, 1, 1, 1], 2),
    ([1, 100, 1, 100, 1], 3), ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 1), ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 10), ([7, 2, 5, 10, 8], 2), ([1, 2, 3, 1, 1], 4),
    ([10, 5, 1, 1, 1, 5, 10], 3), ([2, 2, 2, 2, 2, 2], 4), ([1] * 10, 3),
]
for w, d in edge:
    check(w, d)
n = 40000
assert Solution().shipWithinDays([3] * n, 1) == 3 * n and Solution().shipWithinDays([3] * n, n) == 3
print(len(edge), "edge case + n = 40000 (days = 1 và days = n): đúng")

16 edge case + n = 40000 (days = 1 và days = n): đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def need_buggy(weights, cap, start_days=1, strict=False, drop=False):
    d, load = start_days, 0
    for w in weights:
        if (load + w >= cap) if strict else (load + w > cap):
            d += 1
            load = 0
            if drop:
                continue                              # BUG: bỏ mất gói khi mở ngày mới
        load += w
    return d


def bug_lo_one(weights, days):                        # cận dưới 1 (không dùng max) và đếm ngày kiểu nhét gói nặng vào
    lo, hi = 1, sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        d, load = 1, 0
        for w in weights:
            if load + w > mid and load > 0:           # gói nặng hơn mid vẫn được nhét vào một ngày trống (sai)
                d += 1
                load = 0
            load += w
        if d <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_days_start_zero(weights, days):               # quên ngày đầu tiên (đếm từ 0)
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if need_buggy(weights, mid, start_days=0) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_strict_less(weights, days):                   # dùng >= (chặt) khi kiểm tra tải
    lo, hi = max(weights), sum(weights) + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if need_buggy(weights, mid, strict=True) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_sort(weights, days):                          # sắp xếp lại gói (không được đổi thứ tự)
    return ship_binary(sorted(weights), days)


def bug_average(weights, days):                       # chỉ lấy trung bình
    return -(-sum(weights) // days)


def bug_drop_package(weights, days):                  # mở ngày mới nhưng bỏ mất gói
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if need_buggy(weights, mid, drop=True) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


def bug_return_hi_minus_one(weights, days):           # sai biên vòng nhị phân: lo <= hi và trả hi
    lo, hi = max(weights), sum(weights)
    while lo <= hi:
        mid = (lo + hi) // 2
        if days_needed(weights, mid) <= days:
            hi = mid - 1
        else:
            lo = mid + 1
    return hi


def bug_max_only(weights, days):                      # chỉ trả max(weights)
    return max(weights)


BUGS = [bug_lo_one, bug_days_start_zero, bug_strict_less, bug_sort, bug_average, bug_drop_package, bug_return_hi_minus_one, bug_max_only]
pool = [(list(a), d) for n in range(1, 6) for a in itertools.product(range(1, 5), repeat=n) for d in range(1, n + 1)]
for bug in BUGS:
    found = next(((w, d) for w, d in pool if bug(w, d) != ship_binary(w, d)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:24s} sai trên weights={found[0]}, days={found[1]}: ra {bug(*found)}, đúng {ship_binary(*found)}")

bug_lo_one               sai trên weights=[2], days=1: ra 1, đúng 2
bug_days_start_zero      sai trên weights=[1, 1], days=1: ra 1, đúng 2
bug_strict_less          sai trên weights=[1], days=1: ra 2, đúng 1
bug_sort                 sai trên weights=[1, 2, 1], days=2: ra 2, đúng 3
bug_average              sai trên weights=[1, 3], days=2: ra 2, đúng 3
bug_drop_package         sai trên weights=[1, 1, 1], days=2: ra 1, đúng 2
bug_return_hi_minus_one  sai trên weights=[1], days=1: ra 0, đúng 1
bug_max_only             sai trên weights=[1, 1], days=1: ra 1, đúng 2


### D. Vét cạn mọi mảng nhỏ với oracle DP

Oracle `ship_dp` (quy hoạch động tách mảng; không dùng tham lam hay nhị phân). Mọi mảng độ dài `1..6` với giá trị trong `[1, 4]` (`4 + 16 + 64 + 256 + 1024 + 4096 = 5 460` mảng) và mọi `days ∈ [1, n]`: `~ 5 460 × 3.5` ca. Ngoài ra mảng giá trị `[1, 6]` độ dài `≤ 5`.

In [21]:
t0 = time.perf_counter()
cnt = 0
for hi_val, max_n in ((4, 6), (6, 5)):
    for n in range(1, max_n + 1):
        for arr in itertools.product(range(1, hi_val + 1), repeat=n):
            w = list(arr)
            for d in range(1, n + 1):
                want = ship_dp(w, d)
                for name in IMPLS:
                    assert IMPLS[name](w, d) == want, (name, w, d, IMPLS[name](w, d), want)
                assert valid_capacity(w, d, want)
                cnt += 1
print(cnt, "ca (mảng ≤ 6 phần tử trong [1,4], ≤ 5 phần tử trong [1,6]; mọi days):", len(IMPLS), "cài đặt khớp oracle DP;", round(time.perf_counter() - t0, 1), "s")

75738 ca (mảng ≤ 6 phần tử trong [1,4], ≤ 5 phần tử trong [1,6]; mọi days): 11 cài đặt khớp oracle DP; 2.9 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách nhanh khớp nhau; (2) thoả điều kiện tối ưu `days_needed(ans) ≤ days < days_needed(ans − 1)`; (3) `max ≤ ans ≤ sum`; (4) `ans ≥ ceil(sum / days)`; (5) **đơn điệu theo `days`**: tăng `days` không làm tăng đáp án; (6) nhân mọi trọng số với `k` nhân đáp án với `k`; (7) đảo ngược mảng giữ nguyên đáp án (đối xứng); (8) với `n ≤ 12` khớp DP.

In [22]:
rng = random.Random(20240111)
t0 = time.perf_counter()
for it in range(300):
    n = rng.choice([1, 2, 5, 12, 50, 300, 2000])
    span = rng.choice([1, 3, 10, 500])
    w = [rng.randint(1, span) for _ in range(n)]
    d = rng.randint(1, n)
    res = {name: IMPLS[name](w, d) for name in fast}
    assert len(set(res.values())) == 1, res                                                      # (1)
    ans = res["Solution"]
    assert valid_capacity(w, d, ans)                                                            # (2)
    assert max(w) <= ans <= sum(w)                                                              # (3)
    assert ans >= -(-sum(w) // d)                                                               # (4)
    if d < n:
        assert Solution().shipWithinDays(w, d + 1) <= ans                                       # (5)
    assert Solution().shipWithinDays([3 * x for x in w], d) == 3 * ans                          # (6)
    assert Solution().shipWithinDays(w[::-1], d) == ans                                         # (7)
    if n <= 12:
        assert ship_dp(w, d) == ans                                                             # (8)
print("300 ca ngẫu nhiên (n tới 2000): 8 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

300 ca ngẫu nhiên (n tới 2000): 8 tính chất đúng; 0.6 s


### F. Giới hạn

`n = 5·10^4`, `weights[i] ∈ [1, 500]`; `days = 1`, `days = n`, `days` trung bình; mọi gói `500`; mọi gói `1`; xen kẽ `1` và `500`.

In [23]:
t0 = time.perf_counter()
n = 50000
rng = random.Random(10111)
cases = [
    ([rng.randint(1, 500) for _ in range(n)], 1), ([rng.randint(1, 500) for _ in range(n)], n), ([rng.randint(1, 500) for _ in range(n)], 7),
    ([500] * n, 100), ([1] * n, 999), ([1, 500] * (n // 2), 123), ([rng.randint(1, 500) for _ in range(n)], 25000),
]
for w, d in cases:
    res = {name: IMPLS[name](w, d) for name in fast}
    assert len(set(res.values())) == 1, (d, res)
    ans = res["Solution"]
    assert valid_capacity(w, d, ans)
    print(f"  days = {d:6d}: đáp án = {ans}")
assert Solution().shipWithinDays([500] * n, 100) == 500 * (n // 100)
assert Solution().shipWithinDays([rng.randint(1, 500) for _ in range(n)], n) <= 500
print("n = 5·10^4: các cài đặt nhanh khớp nhau và thoả điều kiện tối ưu;", round(time.perf_counter() - t0, 1), "s")

  days =      1: đáp án = 12476297


  days =  50000: đáp án = 500


  days =      7: đáp án = 1785691


  days =    100: đáp án = 250000
  days =    999: đáp án = 51


  days =    123: đáp án = 102203


  days =  25000: đáp án = 674
n = 5·10^4: các cài đặt nhanh khớp nhau và thoả điều kiện tối ưu; 3.1 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cận dưới là `1` thay vì `max(weights)`

In [24]:
w, d = [5], 1
print("bug:", bug_lo_one(w, d), "| đúng:", Solution().shipWithinDays(w, d))
assert bug_lo_one(w, d) < 5 and Solution().shipWithinDays(w, d) == 5

bug: 1 | đúng: 5


**Vì sao sai:** với `c < max(weights)` có gói **không thể chở** dù một mình; hàm đếm ngày (nếu nhét gói nặng vào một ngày trống) trả "đủ" nhầm. **Đúng:** `lo = max(weights)`.

### Lỗi 2: đếm ngày bắt đầu từ `0`

In [25]:
w, d = [3, 2, 2, 4, 1, 4], 3
print("bug:", bug_days_start_zero(w, d), "| đúng:", Solution().shipWithinDays(w, d))
assert bug_days_start_zero(w, d) != 6

bug: 5 | đúng: 6


**Vì sao sai:** ngày đầu tiên đã dùng ngay khi có gói đầu tiên; đếm `d = 0` ban đầu làm số ngày ít hơn thực tế 1 ⇒ chấp nhận sức chứa quá nhỏ.

### Lỗi 3: kiểm tra tải bằng `>=` (chặt) thay vì `>`

In [26]:
w, d = [4, 4], 1
print("bug:", bug_strict_less(w, d), "| đúng:", Solution().shipWithinDays(w, d))
assert bug_strict_less(w, d) == 9 and Solution().shipWithinDays(w, d) == 8

bug: 9 | đúng: 8


**Vì sao sai:** sức chứa `c` cho phép tải **đúng bằng** `c`. Điều kiện sang ngày mới là `load + w > cap` (không phải `>=`).

### Lỗi 4: sắp xếp lại các gói

In [27]:
w, d = [3, 2, 2, 4, 1, 4], 3
print("bug:", bug_sort(w, d), "| đúng:", Solution().shipWithinDays(w, d))
w2, d2 = [1, 100, 1, 100, 1], 3
print("bug:", bug_sort(w2, d2), "| đúng:", Solution().shipWithinDays(w2, d2))
assert bug_sort(w2, d2) == 100 and Solution().shipWithinDays(w2, d2) == 101      # [1,100,1,100,1]: không thể gộp hai gói 100 nên 3 ngày cần 101

bug: 7 | đúng: 6
bug: 100 | đúng: 101


**Vì sao sai:** đề yêu cầu chất **theo thứ tự trên băng chuyền** (đoạn liên tiếp). Sắp xếp phá ràng buộc.

### Lỗi 5: mở ngày mới nhưng bỏ mất gói

In [28]:
w, d = [3, 2, 2, 4, 1, 4], 3
print("bug:", bug_drop_package(w, d), "| đúng:", Solution().shipWithinDays(w, d))
assert bug_drop_package(w, d) != 6

bug: 5 | đúng: 6


**Vì sao sai:** khi mở ngày mới gói hiện tại phải được **xếp vào ngày mới** (`load = 0` rồi `load += w`).

### Lỗi 6 (bonus): chỉ lấy trung bình / chỉ lấy max / sai biên nhị phân

In [29]:
w, d = [1, 100, 1, 100, 1], 3
print("trung bình:", bug_average(w, d), "| max:", bug_max_only(w, d), "| biên sai:", bug_return_hi_minus_one(w, d), "| đúng:", Solution().shipWithinDays(w, d))
assert bug_average(w, d) != Solution().shipWithinDays(w, d) and bug_max_only(w, d) != Solution().shipWithinDays(w, d)

trung bình: 68 | max: 100 | biên sai: 100 | đúng: 101


**Vì sao sai:** đáp án phụ thuộc cả vào **thứ tự** lẫn `max`; trung bình chỉ là cận dưới. Còn với vòng `while lo <= hi` trả `hi` thì lệch một đơn vị.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tối thiểu hoá giá trị lớn nhất / tối đa hoá giá trị nhỏ nhất" với điều kiện khả thi **đơn điệu** → **tìm nhị phân trên đáp án** + kiểm tra khả thi tham lam.

**Mẹo:**

1. Xác định miền đáp án `[lo, hi]`: `lo = max(weights)`, `hi = sum(weights)`.
2. Khuôn nhị phân tìm "nhỏ nhất thoả": `if ok(mid): hi = mid else: lo = mid + 1`.
3. Hàm kiểm tra tham lam một lượt, đếm ngày bắt đầu từ `1`.
4. Đơn điệu: sức chứa lớn không bao giờ tệ hơn.
5. Oracle độc lập: DP tách mảng min–max.
6. Kiểm tra tối ưu: `ans` đủ và `ans − 1` không đủ.

**Họ bài cùng pattern:** Split Array Largest Sum (410), Koko Eating Bananas (875), Minimum Number of Days to Make m Bouquets (1482), Smallest Divisor Given a Threshold (1283), Minimize Max Distance to Gas Station (774, Premium), Magnetic Force Between Two Balls (1552).

## 15. Biến thể

1. **Split Array Largest Sum (410):** chia mảng thành `k` đoạn liên tiếp tối thiểu hoá tổng đoạn lớn nhất (cùng bài toán, `days = k`).
2. **Lịch chất hàng cụ thể:** trả về từng ngày chở những gói nào.
3. **Số ngày tối thiểu với sức chứa cho trước** (`days_needed`).

Cài đặt và kiểm chứng:

In [30]:
def split_array_largest_sum(nums, k):
    return Solution().shipWithinDays(nums, k)


def make_schedule(weights, days):
    cap = Solution().shipWithinDays(weights, days)
    return schedule(weights, cap), cap


def min_days(weights, cap):
    return days_needed(weights, cap)


rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 9)
    w = [rng.randint(1, 20) for _ in range(n)]
    k = rng.randint(1, n)
    # (1) khớp DP độc lập (đúng định nghĩa 410: chia thành k đoạn không rỗng — dùng ≤ k đoạn cho kết quả như nhau)
    assert split_array_largest_sum(w, k) == ship_dp(w, k)
    # (2) lịch cụ thể hợp lệ
    sched, cap = make_schedule(w, k)
    assert len(sched) <= k and all(sum(day) <= cap for day in sched) and [x for day in sched for x in day] == w
    assert max(sum(day) for day in sched) == cap                                      # sức chứa nhỏ nhất => có một ngày chở đúng cap
    # (3) min_days là cực tiểu: với mọi cách chia thành m đoạn mỗi đoạn <= cap có m >= min_days
    c2 = rng.randint(max(w), sum(w))
    # số ngày tối thiểu với sức chứa c2 theo DP độc lập: nhỏ nhất m sao cho tối thiểu hoá tổng đoạn lớn nhất với m đoạn <= c2
    m = next(m for m in range(1, len(w) + 1) if ship_dp(w, m) <= c2)
    assert min_days(w, c2) == m
print("Split Array Largest Sum, lịch chất hàng, số ngày tối thiểu: khớp DP / kiểm tra hợp lệ")

Split Array Largest Sum, lịch chất hàng, số ngày tối thiểu: khớp DP / kiểm tra hợp lệ


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử tăng dần | `c = max, max+1, …` | `O(n·Δ)` | `O(1)` | oracle |
| 2. Nhị phân + tham lam | kiểm tra một lượt | `O(n log S)` | `O(1)` | **đáp án chính** |
| 3. Nhị phân + `bisect` | nhảy theo tổng tiền tố | `O(n + days·log n·log S)` | `O(n)` | `days` nhỏ |
| 4. Galloping + nhị phân | nhân đôi rồi nhị phân | `O(n log ans)` | `O(1)` | không biết biên trên |
| 5. DP tách mảng | min–max theo đoạn | `O(days·n²)` | `O(days·n)` | oracle độc lập |

Mọi cách khớp oracle DP trên toàn bộ mảng nhỏ (mục 12D) và khớp nhau tới `n = 5·10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().shipWithinDays([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), Solution().shipWithinDays([3, 2, 2, 4, 1, 4], 3), Solution().shipWithinDays([1, 2, 3, 1, 1], 4),
      Solution().shipWithinDays([5], 1), Solution().shipWithinDays([1, 1, 1], 1))

15 6 3 5 3


### Bài 2 (tính tay)

`weights = [7, 2, 5, 10, 8]`, `days = 2`. (a) Cận `lo`, `hi`. (b) Chạy tham lam với `c = 15` (mấy ngày?). (c) Tìm đáp án. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm số ngày cần với sức chứa `cap`:

In [32]:
def need_todo(weights, cap):
    d, load = 1, 0
    for w in weights:
        pass            # TODO 1: nếu load + w > cap thì d += 1 và load = 0
        pass            # TODO 2: load += w
    return d


# Khi làm xong, bỏ comment để kiểm tra:
# assert need_todo([3, 2, 2, 4, 1, 4], 6) == 3 and need_todo([3, 2, 2, 4, 1, 4], 5) == 4

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **galloping** (nhân đôi `c` rồi nhị phân):

In [33]:
class MySolution:
    def shipWithinDays(self, weights, days):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().shipWithinDays(list(w), d) == want for (w, d), want in examples)

### Bài 5 (mini challenge): lịch chất hàng cụ thể

Viết `plan(weights, days)` trả về danh sách các ngày (mỗi ngày là danh sách các gói), dùng **sức chứa nhỏ nhất**; kiểm: số ngày `≤ days`, mỗi ngày `≤ sức chứa`, ghép lại ra đúng `weights` theo thứ tự.

In [34]:
def plan(weights, days):
    pass            # TODO


# Khi làm xong:
# p = plan([3, 2, 2, 4, 1, 4], 3)
# assert len(p) <= 3 and [x for day in p for x in day] == [3, 2, 2, 4, 1, 4] and max(sum(day) for day in p) <= 6

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1, 2, 3 cho `15`, `6`, `3`.
- Mức 2: một gói `[5]`, `days = 1` ⇒ `5`; ba gói `1` trong một ngày ⇒ `3`.
- Mức 3: `15 6 3 5 3`.

**Bài 2**

- Mức 1: `lo = 10`, `hi = 32`.
- Mức 2: với `c = 15`: `[7,2,5]`, `[10]`, `[8]` ⇒ 3 ngày (không đủ cho 2 ngày).
- Mức 3: đáp án `18` với lịch `[7,2,5]` và `[10,8]` (tổng `14` và `18`); `c = 17` không đủ vì `[7,2,5]` rồi `10 + 8 = 18 > 17`.

**Bài 3**

- Mức 1: `if load + w > cap: d += 1; load = 0`.
- Mức 2: sau đó `load += w`.
- Mức 3: `d` bắt đầu bằng `1`.

**Bài 4**

- Mức 1: `c = lo = max(weights)`.
- Mức 2: `while need(c) > days: lo = c + 1; c *= 2`.
- Mức 3: nhị phân trong `[lo, c]`.

**Bài 5**

- Mức 1: tính `cap = shipWithinDays`.
- Mức 2: tham lam: mở ngày mới khi `load + w > cap`.
- Mức 3: nhớ thêm ngày cuối sau vòng lặp.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().shipWithinDays([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), Solution().shipWithinDays([3, 2, 2, 4, 1, 4], 3), Solution().shipWithinDays([1, 2, 3, 1, 1], 4),
        Solution().shipWithinDays([5], 1), Solution().shipWithinDays([1, 1, 1], 1))
print(vals)
assert vals == (15, 6, 3, 5, 3)
print("Bài 1 OK")

(15, 6, 3, 5, 3)
Bài 1 OK


### Bài 2

In [36]:
w, d = [7, 2, 5, 10, 8], 2
print("lo, hi =", max(w), sum(w), "| c = 15:", schedule(w, 15), days_needed(w, 15), "ngày")
ans = Solution().shipWithinDays(w, d)
print("đáp án:", ans, "| lịch:", schedule(w, ans))
assert (max(w), sum(w)) == (10, 32) and days_needed(w, 15) == 3 and ans == 18 == ship_dp(w, d)
assert schedule(w, 18) == [[7, 2, 5], [10, 8]]
print("Bài 2 OK")

lo, hi = 10 32 | c = 15: [[7, 2, 5], [10], [8]] 3 ngày
đáp án: 18 | lịch: [[7, 2, 5], [10, 8]]
Bài 2 OK


### Bài 3

In [37]:
def need_answer(weights, cap):
    d, load = 1, 0
    for w in weights:
        if load + w > cap:
            d += 1
            load = 0
        load += w
    return d


assert need_answer([3, 2, 2, 4, 1, 4], 6) == 3 and need_answer([3, 2, 2, 4, 1, 4], 5) == 4
rng = random.Random(3)
for _ in range(300):
    w = [rng.randint(1, 9) for _ in range(rng.randint(1, 20))]
    c = rng.randint(max(w), sum(w))
    assert need_answer(w, c) == days_needed(w, c)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def shipWithinDays(self, weights, days):
        def need(cap):
            d, load = 1, 0
            for w in weights:
                if load + w > cap:
                    d += 1
                    load = 0
                load += w
            return d

        lo = c = max(weights)
        while need(c) > days:
            lo = c + 1
            c *= 2
        hi = c
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= days:
                hi = mid
            else:
                lo = mid + 1
        return lo


assert all(MySolutionAnswer().shipWithinDays(list(w), d) == want for (w, d), want in examples)
rng = random.Random(4)
for _ in range(300):
    w = [rng.randint(1, 20) for _ in range(rng.randint(1, 9))]
    d = rng.randint(1, len(w))
    assert MySolutionAnswer().shipWithinDays(w, d) == ship_dp(w, d)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def plan(weights, days):
    cap = Solution().shipWithinDays(weights, days)
    out, cur, load = [], [], 0
    for w in weights:
        if load + w > cap:
            out.append(cur)
            cur, load = [], 0
        cur.append(w)
        load += w
    out.append(cur)
    return out


p = plan([3, 2, 2, 4, 1, 4], 3)
assert len(p) <= 3 and [x for day in p for x in day] == [3, 2, 2, 4, 1, 4] and max(sum(day) for day in p) <= 6
rng = random.Random(5)
for _ in range(300):
    w = [rng.randint(1, 20) for _ in range(rng.randint(1, 12))]
    d = rng.randint(1, len(w))
    pl = plan(w, d)
    cap = ship_dp(w, d)
    assert len(pl) <= d and [x for day in pl for x in day] == w and max(sum(day) for day in pl) == cap
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tối thiểu hoá giá trị lớn nhất với khả thi đơn điệu → tìm nhị phân trên đáp án):**

```python
class Solution:
    def shipWithinDays(self, weights: List[int], days: int) -> int:
        def need(cap):
            d, load = 1, 0
            for w in weights:
                if load + w > cap:
                    d += 1
                    load = 0
                load += w
            return d

        lo, hi = max(weights), sum(weights)
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= days:
                hi = mid
            else:
                lo = mid + 1
        return lo
```

**Checklist trước khi submit:**

- [ ] `lo = max(weights)`, `hi = sum(weights)`?
- [ ] Đếm ngày bắt đầu từ `1`; sang ngày mới khi `load + w > cap`; sau đó `load = 0` rồi cộng `w`?
- [ ] Không sắp xếp lại `weights` (đoạn liên tiếp, giữ thứ tự)?
- [ ] Khuôn nhị phân `if ok(mid): hi = mid else: lo = mid + 1` (trả `lo`)?
- [ ] Đã thử `days = 1`, `days = n`, một gói, ví dụ 1–3?
- [ ] Biết `ans` đủ và `ans − 1` không đủ (kiểm tối ưu)?
- [ ] Đúng tên `class Solution`, `shipWithinDays(self, weights, days)`, không in debug.